# Dynamic Factor Model for Nowcasting

This notebook demonstrates how to use **Dynamic Factor Models (DFM)** for nowcasting GDP growth
using mixed-frequency economic indicators.

**Reference**: Giannone, D., Reichlin, L., & Small, D. (2008). "Nowcasting: The real-time informational
content of macroeconomic data." *Journal of Monetary Economics*, 55(4), 665-676.

The DFM approach extracts common latent factors from a large panel of indicators observed at different
frequencies, using a state-space representation estimated via the **EM algorithm** with
**Kalman filter/smoother**.

In [ ]:
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from forecastbox.nowcasting import DFMNowcaster, NewsDecomposition

# Add helpers path
sys.path.insert(0, "../../utils")
from helpers import load_mixed_freq, load_macro_brazil, simulate_ragged_edge

warnings.filterwarnings("ignore")
np.random.seed(42)

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

## 1. The Nowcasting Problem

Nowcasting is the prediction of the **present** or very near future. GDP is released with a
significant delay (typically 1-3 months after the quarter ends), but monthly indicators such as
industrial production, retail sales, and confidence indices are available much sooner.

The key challenge is the **ragged edge**: at any point in time, different indicators have
different amounts of data available, creating an unbalanced panel with missing observations
at the end of each series.

In [ ]:
# Load the mixed-frequency dataset
data = load_mixed_freq()
print(f"Dataset shape: {data.shape}")
print(f"Date range: {data.index[0]} to {data.index[-1]}")
print(f"\nColumns: {list(data.columns)}")
print(f"\nMissing values per column:")
print(data.isna().sum())

# Visualize the ragged edge pattern
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for ax, col in zip(axes.flat, data.columns):
    series = data[col].dropna()
    ax.plot(series.index, series.values, "b-", linewidth=1.2)
    ax.set_title(col.replace("_", " ").title(), fontsize=12)
    ax.grid(True, alpha=0.3)
    # Mark NaN periods
    nan_mask = data[col].isna()
    if nan_mask.any():
        for idx in data.index[nan_mask]:
            ax.axvline(idx, color="red", alpha=0.1, linewidth=0.5)

fig.suptitle("Mixed-Frequency Data with Ragged Edge", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Show the ragged edge pattern explicitly
print("\nRagged Edge Pattern (last 6 months):")
print(data.tail(6).to_string())

## 2. Dynamic Factor Model

The DFM represents the co-movement of a panel of $N$ indicators $x_t = (x_{1t}, \ldots, x_{Nt})'$
through a small number of latent factors $f_t$:

**Observation equation (measurement):**
$$x_t = \Lambda f_t + e_t, \quad e_t \sim N(0, R)$$

**State equation (transition):**
$$f_t = A_1 f_{t-1} + A_2 f_{t-2} + \ldots + A_p f_{t-p} + u_t, \quad u_t \sim N(0, Q)$$

Where:
- $\Lambda$ is the matrix of **factor loadings** (how each indicator loads on the factors)
- $A_1, \ldots, A_p$ are the **VAR coefficients** for factor dynamics
- $R$ is the **observation noise** covariance (diagonal, idiosyncratic)
- $Q$ is the **state noise** covariance

The **Kalman filter** handles missing data naturally: when an observation is missing,
the filter skips the update step for that variable, using only the prediction.

For **mixed-frequency** data, quarterly variables use the **Mariano-Murasawa (2003)**
triangular accumulator in the state space, linking the quarterly flow to monthly factors.

In [ ]:
# Define frequency map: which variables are monthly vs quarterly
frequency_map = {
    "industrial_production": "M",
    "retail_sales": "M",
    "confidence_index": "M",
    "gdp_growth": "Q",
}

# Create and fit DFM with 1 factor
dfm = DFMNowcaster(
    n_factors=1,
    factor_lags=2,
    frequency_map=frequency_map,
    aggregation="sum",
    em_iterations=100,
    em_tol=1e-6,
)

dfm.fit(data)
print(dfm)

# Show factor loadings
print("\nFactor Loadings:")
print(dfm.loadings())

## 3. Extracting Factors

The latent factor captures the common dynamics across all indicators. A positive loading
means the indicator moves with the factor; a negative loading means it moves against.

We can visualize the estimated factor against the observed GDP growth to see how well
the factor tracks the target variable.

In [ ]:
# Extract estimated factors
factors = dfm.factors()
print(f"Factors shape: {factors.shape}")
print(f"\nFactor statistics:")
print(factors.describe())

# Plot factor vs GDP growth
fig, ax1 = plt.subplots(figsize=(14, 6))

# Plot factor on left axis
color1 = "steelblue"
ax1.plot(factors.index, factors["factor_1"], color=color1, linewidth=2, label="Latent Factor 1")
ax1.set_xlabel("Date")
ax1.set_ylabel("Factor Value", color=color1)
ax1.tick_params(axis="y", labelcolor=color1)

# Plot GDP on right axis
ax2 = ax1.twinx()
color2 = "darkorange"
gdp_obs = data["gdp_growth"].dropna()
ax2.scatter(gdp_obs.index, gdp_obs.values, color=color2, s=40, zorder=5, label="GDP Growth (quarterly)")
ax2.set_ylabel("GDP Growth", color=color2)
ax2.tick_params(axis="y", labelcolor=color2)

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

ax1.set_title("Estimated Latent Factor vs GDP Growth", fontsize=14, fontweight="bold")
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Nowcasting GDP

The key advantage of the DFM approach is its ability to generate a nowcast even when the
data panel has a **ragged edge** — i.e., different indicators have data available up to
different points in time.

We simulate this by removing the last few observations from some indicators.

In [ ]:
# Simulate ragged edge: some indicators missing more recent data
ragged_data = simulate_ragged_edge(data, {
    "industrial_production": 1,   # 1 month missing
    "retail_sales": 2,            # 2 months missing
    "confidence_index": 0,        # fully available
    "gdp_growth": 3,              # 3 months missing (typical for GDP)
})

print("Ragged edge pattern (last 6 months):")
print(ragged_data.tail(6).to_string())

# Fit DFM on ragged-edge data and nowcast
dfm_ragged = DFMNowcaster(
    n_factors=1,
    factor_lags=2,
    frequency_map=frequency_map,
    aggregation="sum",
    em_iterations=100,
)
dfm_ragged.fit(ragged_data)

nowcast = dfm_ragged.nowcast(target="gdp_growth")
print(f"\nGDP Nowcast: {nowcast.point[0]:.4f}")
print(f"80% CI: [{nowcast.lower_80[0]:.4f}, {nowcast.upper_80[0]:.4f}]")
print(f"95% CI: [{nowcast.lower_95[0]:.4f}, {nowcast.upper_95[0]:.4f}]")
print(f"Model: {nowcast.model_name}")

## 5. News Decomposition

A powerful feature of the DFM framework is the ability to decompose nowcast **revisions**
into contributions from each newly released data point. This follows
**Banbura & Modugno (2014)**.

When new data arrives, the nowcast revision is:

$$\Delta \hat{y}_{t|\Omega_{new}} - \hat{y}_{t|\Omega_{old}} = \sum_i w_i \cdot (x_i^{new} - E[x_i | \Omega_{old}])$$

Where:
- $w_i$ are the **weights** (sensitivity of the nowcast to indicator $i$)
- $x_i^{new} - E[x_i | \Omega_{old}]$ is the **news** (surprise in the new release)

In [ ]:
# Create old and new information sets to demonstrate news decomposition
# Old: more missing data
old_data = simulate_ragged_edge(data, {
    "industrial_production": 3,
    "retail_sales": 3,
    "confidence_index": 2,
    "gdp_growth": 4,
})

# New: some indicators updated
new_data = simulate_ragged_edge(data, {
    "industrial_production": 1,
    "retail_sales": 2,
    "confidence_index": 0,
    "gdp_growth": 4,
})

# Perform news decomposition
news_decomp = NewsDecomposition(dfm)
result = news_decomp.decompose(old_data, new_data, target="gdp_growth")

# Print summary
print(result.summary())

# Plot contributions as bar chart
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

result.plot_contributions(ax=axes[0])
axes[0].set_title("News Contributions to Nowcast Revision", fontsize=12)

result.plot_waterfall(ax=axes[1])
axes[1].set_title("Waterfall: Old Nowcast \u2192 New Nowcast", fontsize=12)

plt.tight_layout()
plt.show()

## 6. Pseudo Real-Time Exercise

To evaluate nowcasting performance, we simulate a **pseudo real-time** exercise:
at each point in time, we only use data that would have been available, then generate
a nowcast and compare it to the actual GDP release.

This mimics how the model would perform in practice, accounting for the ragged edge.

In [ ]:
# Pseudo real-time exercise: simulate sequential data arrival
gdp_dates = data["gdp_growth"].dropna().index
# Use the last 8 quarters for evaluation
eval_dates = gdp_dates[-8:]

nowcast_results = []

for eval_date in eval_dates:
    # Simulate data available at different lags within the quarter
    for months_before_release in [3, 2, 1]:
        # Create ragged-edge data as if we were `months_before_release` months
        # before the GDP release
        cutoff_idx = data.index.get_loc(eval_date) - months_before_release
        if cutoff_idx < 12:
            continue

        available_data = data.iloc[:cutoff_idx + 1].copy()
        # GDP is not yet available for this quarter
        available_data.loc[eval_date:, "gdp_growth"] = np.nan

        # Fit and nowcast
        rt_dfm = DFMNowcaster(
            n_factors=1,
            factor_lags=2,
            frequency_map=frequency_map,
            aggregation="sum",
            em_iterations=50,
        )
        try:
            rt_dfm.fit(available_data)
            fc = rt_dfm.nowcast(target="gdp_growth")
            actual = data.loc[eval_date, "gdp_growth"]

            nowcast_results.append({
                "quarter": eval_date,
                "months_before": months_before_release,
                "nowcast": fc.point[0],
                "actual": actual,
                "error": fc.point[0] - actual,
            })
        except Exception:
            pass

results_df = pd.DataFrame(nowcast_results)
print("Pseudo Real-Time Results:")
print(results_df.to_string(index=False))

# Plot nowcast evolution for each quarter
fig, ax = plt.subplots(figsize=(14, 6))

for quarter in results_df["quarter"].unique():
    qdata = results_df[results_df["quarter"] == quarter].sort_values("months_before", ascending=False)
    label = quarter.strftime("%Y-Q%q") if hasattr(quarter, "strftime") else str(quarter)
    ax.plot(qdata["months_before"], qdata["nowcast"], "o-", label=f"{quarter.year}Q{(quarter.month-1)//3+1}")
    ax.axhline(qdata["actual"].iloc[0], linestyle="--", alpha=0.3)

ax.set_xlabel("Months Before GDP Release")
ax.set_ylabel("Nowcast Value")
ax.set_title("Nowcast Evolution as Data Arrives", fontsize=14, fontweight="bold")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=9)
ax.invert_xaxis()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# RMSE by horizon
print("\nRMSE by months before release:")
for m in sorted(results_df["months_before"].unique()):
    subset = results_df[results_df["months_before"] == m]
    rmse = np.sqrt(np.mean(subset["error"] ** 2))
    print(f"  {m} months before: RMSE = {rmse:.4f}")

### Exercise 1: DFM with 2 factors for Brazilian macro

Load the `macro_brazil.csv` dataset and fit a DFM with 2 factors. Compare the factors
to GDP growth and inflation. Which indicators load most strongly on each factor?

In [ ]:
# Exercise 1 - Solution: DFM with 2 factors for Brazilian macro data

# Step 1: Load Brazilian macroeconomic data
brazil = load_macro_brazil()
print(f"Brazilian macro dataset: {brazil.shape}")
print(f"Columns: {list(brazil.columns)}")
print(f"Date range: {brazil.index[0]} to {brazil.index[-1]}")
print(f"\nDescriptive statistics:")
print(brazil.describe().round(4))

# Step 2: Prepare data - treat gdp_growth as quarterly (keep only quarter-end months)
brazil_mf = brazil.copy()
# Make GDP quarterly: only keep values at end of each quarter (March, June, Sept, Dec)
quarterly_mask = ~brazil_mf.index.month.isin([3, 6, 9, 12])
brazil_mf.loc[quarterly_mask, "gdp_growth"] = np.nan

print(f"\nAfter making GDP quarterly:")
print(f"  GDP non-null observations: {brazil_mf['gdp_growth'].notna().sum()}")
print(f"  Monthly indicators: {brazil_mf[['inflation', 'interest_rate', 'unemployment', 'exchange_rate']].notna().sum().to_dict()}")

In [ ]:
# Step 3: Define frequency map for all 5 variables
brazil_freq_map = {
    "inflation": "M",
    "interest_rate": "M",
    "unemployment": "M",
    "exchange_rate": "M",
    "gdp_growth": "Q",
}

# Step 4: Create and fit DFM with 2 factors
dfm_brazil = DFMNowcaster(
    n_factors=2,
    factor_lags=2,
    frequency_map=brazil_freq_map,
    aggregation="sum",
    em_iterations=100,
    em_tol=1e-6,
)
dfm_brazil.fit(brazil_mf)
print(dfm_brazil)

# Step 5: Examine factor loadings
loadings = dfm_brazil.loadings()
print("\nFactor Loadings:")
print(loadings.to_string())
print("\n--- Interpretation ---")
for col in loadings.index:
    f1 = loadings.loc[col, "factor_1"]
    f2 = loadings.loc[col, "factor_2"]
    dominant = "Factor 1" if abs(f1) > abs(f2) else "Factor 2"
    print(f"  {col:20s}: F1={f1:+.4f}, F2={f2:+.4f} -> Loads mainly on {dominant}")

In [ ]:
# Step 6: Extract factors and compute correlations with GDP and inflation
factors_brazil = dfm_brazil.factors()
print(f"Factors shape: {factors_brazil.shape}")

# Compute correlations with GDP growth and inflation
gdp_quarterly = brazil_mf["gdp_growth"].dropna()
inflation_monthly = brazil_mf["inflation"].dropna()

# Align factors with GDP (quarterly dates)
common_gdp_idx = factors_brazil.index.intersection(gdp_quarterly.index)
corr_gdp_f1 = np.corrcoef(factors_brazil.loc[common_gdp_idx, "factor_1"], gdp_quarterly.loc[common_gdp_idx])[0, 1]
corr_gdp_f2 = np.corrcoef(factors_brazil.loc[common_gdp_idx, "factor_2"], gdp_quarterly.loc[common_gdp_idx])[0, 1]

# Align factors with inflation (monthly dates)
common_inf_idx = factors_brazil.index.intersection(inflation_monthly.index)
corr_inf_f1 = np.corrcoef(factors_brazil.loc[common_inf_idx, "factor_1"], inflation_monthly.loc[common_inf_idx])[0, 1]
corr_inf_f2 = np.corrcoef(factors_brazil.loc[common_inf_idx, "factor_2"], inflation_monthly.loc[common_inf_idx])[0, 1]

print("\n--- Correlations ---")
print(f"  Factor 1 vs GDP growth:  {corr_gdp_f1:+.4f}")
print(f"  Factor 2 vs GDP growth:  {corr_gdp_f2:+.4f}")
print(f"  Factor 1 vs Inflation:   {corr_inf_f1:+.4f}")
print(f"  Factor 2 vs Inflation:   {corr_inf_f2:+.4f}")
print("\nInterpretation:")
print(f"  Factor 1 is more correlated with {'GDP' if abs(corr_gdp_f1) > abs(corr_inf_f1) else 'Inflation'}")
print(f"  Factor 2 is more correlated with {'GDP' if abs(corr_gdp_f2) > abs(corr_inf_f2) else 'Inflation'}")

In [ ]:
# Step 7: Plot both factors vs GDP growth and Inflation
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Top: Factor 1 and Factor 2 vs GDP growth
ax1 = axes[0]
ax1.plot(factors_brazil.index, factors_brazil["factor_1"], color="steelblue",
         linewidth=2, label=f"Factor 1 (corr={corr_gdp_f1:+.3f})")
ax1.plot(factors_brazil.index, factors_brazil["factor_2"], color="forestgreen",
         linewidth=2, label=f"Factor 2 (corr={corr_gdp_f2:+.3f})")
ax1.set_ylabel("Factor Value")

ax1_r = ax1.twinx()
ax1_r.scatter(gdp_quarterly.index, gdp_quarterly.values, color="darkorange",
              s=40, zorder=5, label="GDP Growth (Q)")
ax1_r.set_ylabel("GDP Growth", color="darkorange")
ax1_r.tick_params(axis="y", labelcolor="darkorange")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1_r.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")
ax1.set_title("Latent Factors vs GDP Growth", fontsize=13, fontweight="bold")
ax1.grid(True, alpha=0.3)

# Bottom: Factor 1 and Factor 2 vs Inflation
ax2 = axes[1]
ax2.plot(factors_brazil.index, factors_brazil["factor_1"], color="steelblue",
         linewidth=2, label=f"Factor 1 (corr={corr_inf_f1:+.3f})")
ax2.plot(factors_brazil.index, factors_brazil["factor_2"], color="forestgreen",
         linewidth=2, label=f"Factor 2 (corr={corr_inf_f2:+.3f})")
ax2.set_ylabel("Factor Value")

ax2_r = ax2.twinx()
ax2_r.plot(inflation_monthly.index, inflation_monthly.values, color="crimson",
           linewidth=1, alpha=0.7, label="Inflation (M)")
ax2_r.set_ylabel("Inflation", color="crimson")
ax2_r.tick_params(axis="y", labelcolor="crimson")

lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2_r.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc="upper right")
ax2.set_title("Latent Factors vs Inflation", fontsize=13, fontweight="bold")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== Summary ===")
print("The 2-factor DFM for Brazilian macro data reveals:")
print("  - Factor 1 captures the common real-activity component (correlated with GDP)")
print("  - Factor 2 captures the nominal/financial component (correlated with inflation/rates)")
print("  - This separation is typical in macroeconomic factor models")

### Exercise 2: Compare DFM with different numbers of factors

Fit DFM models with 1, 2, and 3 factors on the `mixed_freq.csv` dataset.
Compare their nowcast accuracy using a pseudo real-time exercise.
Does adding more factors improve the nowcast?

In [ ]:
# Exercise 2 - Solution: Compare DFM with 1, 2, 3 factors

# Step 1: Run pseudo real-time exercise for each number of factors
n_factors_list = [1, 2, 3]
all_results = {}

gdp_dates = data["gdp_growth"].dropna().index
eval_dates = gdp_dates[-8:]  # Last 8 quarters

for n_f in n_factors_list:
    print(f"\nRunning pseudo real-time exercise with n_factors={n_f}...")
    factor_results = []

    for eval_date in eval_dates:
        for months_before in [3, 2, 1]:
            cutoff_idx = data.index.get_loc(eval_date) - months_before
            if cutoff_idx < 12:
                continue

            available_data = data.iloc[:cutoff_idx + 1].copy()
            available_data.loc[eval_date:, "gdp_growth"] = np.nan

            rt_dfm = DFMNowcaster(
                n_factors=n_f,
                factor_lags=2,
                frequency_map=frequency_map,
                aggregation="sum",
                em_iterations=50,
            )
            try:
                rt_dfm.fit(available_data)
                fc = rt_dfm.nowcast(target="gdp_growth")
                actual = data.loc[eval_date, "gdp_growth"]
                factor_results.append({
                    "quarter": eval_date,
                    "months_before": months_before,
                    "nowcast": fc.point[0],
                    "actual": actual,
                    "error": fc.point[0] - actual,
                })
            except Exception:
                pass

    all_results[n_f] = pd.DataFrame(factor_results)
    print(f"  Collected {len(factor_results)} nowcasts")

In [ ]:
# Step 2: Compute information criteria (IC) approximation for model selection
# We use the residual variance from the fitted models as a proxy
print("\n" + "=" * 70)
print("Information Criteria (IC) for Factor Selection")
print("=" * 70)

ic_table = []
T = len(data)
n_vars = len([c for c in data.columns if c != "gdp_growth"])

for n_f in n_factors_list:
    # Fit DFM on full data to get fit statistics
    dfm_ic = DFMNowcaster(
        n_factors=n_f,
        factor_lags=2,
        frequency_map=frequency_map,
        aggregation="sum",
        em_iterations=100,
    )
    dfm_ic.fit(data)

    # Use log-likelihood from EM for IC computation
    log_lik = dfm_ic._log_likelihood if hasattr(dfm_ic, "_log_likelihood") else np.nan

    # Number of parameters: n_vars * n_factors (loadings) + n_factors^2 * factor_lags (VAR)
    #                       + n_vars (R diagonal) + n_factors * (n_factors+1)/2 (Q)
    n_params = (n_vars * n_f + n_f**2 * 2 + n_vars +
                n_f * (n_f + 1) // 2)

    # Compute RMSE from pseudo real-time
    res_df = all_results[n_f]
    rmse_overall = np.sqrt(np.mean(res_df["error"] ** 2)) if len(res_df) > 0 else np.nan

    # Approximations of IC
    if not np.isnan(log_lik):
        aic = -2 * log_lik + 2 * n_params
        bic = -2 * log_lik + np.log(T) * n_params
    else:
        # Fallback: use residual variance
        sigma2 = rmse_overall ** 2 if not np.isnan(rmse_overall) else 1.0
        aic = T * np.log(sigma2) + 2 * n_params
        bic = T * np.log(sigma2) + np.log(T) * n_params

    ic_table.append({
        "n_factors": n_f,
        "n_params": n_params,
        "AIC": aic,
        "BIC": bic,
        "RMSE": rmse_overall,
    })

ic_df = pd.DataFrame(ic_table)
print("\nModel Comparison Table:")
print(ic_df.to_string(index=False, float_format="{:.4f}".format))

# Identify best model by each criterion
best_aic = ic_df.loc[ic_df["AIC"].idxmin(), "n_factors"]
best_bic = ic_df.loc[ic_df["BIC"].idxmin(), "n_factors"]
best_rmse = ic_df.loc[ic_df["RMSE"].idxmin(), "n_factors"]
print(f"\nBest by AIC:  {best_aic} factors")
print(f"Best by BIC:  {best_bic} factors")
print(f"Best by RMSE: {best_rmse} factors")

In [ ]:
# Step 3: Detailed RMSE comparison by horizon
print("\n" + "=" * 70)
print("RMSE by Horizon and Number of Factors")
print("=" * 70)

rmse_table = []
for n_f in n_factors_list:
    res_df = all_results[n_f]
    for m in sorted(res_df["months_before"].unique()):
        subset = res_df[res_df["months_before"] == m]
        rmse = np.sqrt(np.mean(subset["error"] ** 2))
        mae = np.mean(np.abs(subset["error"]))
        rmse_table.append({
            "n_factors": n_f,
            "months_before": m,
            "RMSE": rmse,
            "MAE": mae,
            "n_obs": len(subset),
        })

rmse_df = pd.DataFrame(rmse_table)
print("\nDetailed Results:")
print(rmse_df.to_string(index=False, float_format="{:.4f}".format))

# Pivot table for cleaner view
pivot = rmse_df.pivot_table(values="RMSE", index="months_before", columns="n_factors")
pivot.columns = [f"{c} factors" for c in pivot.columns]
print("\nRMSE Pivot Table (rows=horizon, cols=n_factors):")
print(pivot.to_string(float_format="{:.4f}".format))

In [ ]:
# Step 4: Visualize results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Left: RMSE by number of factors (overall)
ax = axes[0]
overall_rmse = [all_results[n_f]["error"].pow(2).mean() ** 0.5 for n_f in n_factors_list]
ax.bar(n_factors_list, overall_rmse, color=["steelblue", "darkorange", "forestgreen"],
       edgecolor="black", alpha=0.8)
ax.set_xlabel("Number of Factors")
ax.set_ylabel("RMSE")
ax.set_title("Overall Nowcast RMSE", fontsize=12, fontweight="bold")
ax.set_xticks(n_factors_list)
for i, v in enumerate(overall_rmse):
    ax.text(n_factors_list[i], v + 0.002, f"{v:.4f}", ha="center", fontsize=10)
ax.grid(True, alpha=0.3, axis="y")

# Middle: RMSE by horizon for each model
ax = axes[1]
colors = {1: "steelblue", 2: "darkorange", 3: "forestgreen"}
for n_f in n_factors_list:
    subset = rmse_df[rmse_df["n_factors"] == n_f]
    ax.plot(subset["months_before"], subset["RMSE"], "o-",
            color=colors[n_f], linewidth=2, markersize=8,
            label=f"{n_f} factor{'s' if n_f > 1 else ''}")
ax.set_xlabel("Months Before Release")
ax.set_ylabel("RMSE")
ax.set_title("RMSE by Horizon", fontsize=12, fontweight="bold")
ax.legend()
ax.invert_xaxis()
ax.grid(True, alpha=0.3)

# Right: IC comparison
ax = axes[2]
x_pos = np.arange(len(n_factors_list))
width = 0.35
# Normalize IC for better visualization
aic_vals = ic_df["AIC"].values
bic_vals = ic_df["BIC"].values
ax.bar(x_pos - width / 2, aic_vals - aic_vals.min(), width, label="AIC (relative)",
       color="steelblue", alpha=0.8, edgecolor="black")
ax.bar(x_pos + width / 2, bic_vals - bic_vals.min(), width, label="BIC (relative)",
       color="coral", alpha=0.8, edgecolor="black")
ax.set_xlabel("Number of Factors")
ax.set_ylabel("IC (relative to minimum)")
ax.set_title("Information Criteria", fontsize=12, fontweight="bold")
ax.set_xticks(x_pos)
ax.set_xticklabels(n_factors_list)
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

plt.suptitle("DFM Factor Selection: Bias-Variance Tradeoff", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("\n=== Discussion ===")
print("Bias-Variance Tradeoff in Factor Selection:")
print("  - 1 factor: Simplest model, may underfit if there are multiple driving forces")
print("  - 2 factors: Captures real-activity and nominal components separately")
print("  - 3 factors: Risk of overfitting with only 4 observed variables")
print(f"\nWith {len(data.columns)} variables, BIC typically favors fewer factors")
print("to penalize complexity, while AIC may select more.")
print("In practice, 1-2 factors usually suffice for small panels.")